# Week 3 · Class 3 — Qdrant & the Retrieval-Enabled Agent

The payoff class. You load your chunks into **Qdrant** (a real vector database with payloads and
filters), build a `retrieve()` function, and swap it into your Week 2 LangGraph agent's `docs`
node. By the end, "what does our PDF say about pricing?" is answered by **live top-k retrieval
with sources cited** — no hardcoded context anywhere.

## Before you start

This notebook is standalone and runs top to bottom on its own. Section 8 rebuilds the Week 2
agent (condensed) so the final wiring actually runs here. If you have your Class 3 agent open in
another tab, you only need `retrieve()` and the new `docs_node` from Sections 6–8.

Get a free API key at [console.groq.com](https://console.groq.com) before Section 2.
Everything else (embeddings, Qdrant) runs locally — no other keys.

## Section 1 — Install packages

In [ ]:
!pip install -q qdrant-client sentence-transformers langchain-core langchain-groq langgraph langchain-community duckduckgo-search

## Section 2 — Groq API key

In [ ]:
import os
import getpass

# Try Colab Secrets first; fall back to a prompt.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
print(llm.invoke("Say 'ready' and nothing else.").content)

## Section 3 — Load your chunks

Same pattern as the last two classes: your Week 2 `data/chunks.json` if present, otherwise a
built-in fallback corpus (with varied `source` values so the filter demos work).

In [ ]:
import json, os

FALLBACK = [
    {"text": "Automobile repair pricing starts at 80 pounds per hour at our garage. Diagnostics are free with any repair.", "source": "garage.txt"},
    {"text": "The 2026 FIFA World Cup is hosted by the USA, Canada, and Mexico, with 48 teams in 12 groups.", "source": "worldcup.wiki"},
    {"text": "Group A of the 2026 World Cup includes Mexico, which plays the opening match at the Azteca.", "source": "worldcup.wiki"},
    {"text": "Our refund policy allows returns within 30 days with a valid receipt. Refunds are processed in 5-7 business days.", "source": "policy.pdf"},
    {"text": "Shipping is free for orders above 50 pounds; otherwise a 4.99 flat fee applies.", "source": "policy.pdf"},
    {"text": "Python 3.13 introduced an experimental free-threaded mode that removes the GIL.", "source": "python-blog"},
    {"text": "A Light in the Attic is a poetry book priced at 51.77 GBP with a three-star rating.", "source": "books.json"},
    {"text": "Tipping the Velvet is a historical fiction book priced at 53.74 GBP.", "source": "books.json"},
]

chunks = None
for path in ["data/chunks.json", "chunks.json"]:
    if os.path.exists(path):
        raw = json.load(open(path))
        chunks = [
            {"text": c, "source": path} if isinstance(c, str) else c
            for c in raw
        ]
        print(f"Loaded {len(chunks)} chunks from {path}")
        break

if chunks is None:
    chunks = FALLBACK
    print(f"No Week 2 file found — using {len(chunks)} fallback chunks.")
    print("Upload data/chunks.json (and merge your scraped JSONs!) to use your own data.")

## Section 4 — Embed and load into Qdrant

Qdrant's three nouns:

- **Collection** — a named set of vectors with one config (like a SQL table)
- **Point** — one entry: `id` + `vector` + `payload` (like a row)
- **Payload** — arbitrary JSON riding with the vector: text, source, page, price… **and filterable**

`QdrantClient(":memory:")` runs the full database inside this process — zero setup. Later, the
*same code* talks to a real server by changing one line: `QdrantClient(url="http://localhost:6333")`.

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

model = SentenceTransformer("all-MiniLM-L6-v2")
client = QdrantClient(":memory:")

COLLECTION = "bootcamp_chunks"

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

texts = [c["text"] for c in chunks]
embs = model.encode(texts, show_progress_bar=True)

points = [
    PointStruct(id=i, vector=embs[i].tolist(), payload=chunks[i])
    for i in range(len(chunks))
]
client.upsert(collection_name=COLLECTION, points=points)

print("Points in collection:", client.count(COLLECTION).count)

## Section 5 — Search: payloads come back with every hit

No more `ids → chunk_store[i]` bookkeeping like FAISS — the text and source ride along.

In [ ]:
q = model.encode("how much does it cost to fix my car?").tolist()

hits = client.query_points(collection_name=COLLECTION, query=q, limit=3).points

for h in hits:
    print(f"{h.score:.3f}  [{h.payload['source']}]  {h.payload['text'][:80]}")

## Section 6 — Filtered search: meaning AND metadata

Class 1 lesson: embeddings can't do "under £20" or "only from the PDF". **Payload filters can.**
The filter is applied *inside* the index — you don't pay to search excluded points.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

# Only chunks whose source is the policy PDF:
hits = client.query_points(
    collection_name=COLLECTION,
    query=model.encode("what are the rules about giving money back?").tolist(),
    query_filter=Filter(must=[
        FieldCondition(key="source", match=MatchValue(value="policy.pdf")),
    ]),
    limit=3,
).points

for h in hits:
    print(f"{h.score:.3f}  [{h.payload['source']}]  {h.payload['text'][:80]}")

### 🔹 Mini-exercise (5 min)

1. Write a filtered query that searches **only** the `books.json` chunks.
2. If your book payloads carry `price_gbp`, add
   `FieldCondition(key="price_gbp", range=Range(lte=52.0))` and search "poetry books" —
   semantic match *and* price cap in one query.
3. What happens with a filter that matches nothing? Check the returned list.

In [ ]:
# TODO: your filtered queries here


## Section 7 — `retrieve()`: the function your agent will call

One addition: a **score threshold**. MiniLM cosine below ~0.35 usually means "the corpus
doesn't actually answer this" — better to admit it than let the LLM improvise.

In [ ]:
SCORE_THRESHOLD = 0.35

def retrieve(question: str, k: int = 3, source: str | None = None) -> list[dict]:
    """Embed the question, search Qdrant, return chunks above the threshold."""
    qfilter = None
    if source:
        qfilter = Filter(must=[FieldCondition(key="source", match=MatchValue(value=source))])
    hits = client.query_points(
        collection_name=COLLECTION,
        query=model.encode(question).tolist(),
        query_filter=qfilter,
        limit=k,
    ).points
    return [
        {"text": h.payload["text"], "source": h.payload["source"], "score": h.score}
        for h in hits
        if h.score >= SCORE_THRESHOLD
    ]

print(retrieve("world cup groups")[0])
print(retrieve("quantum entanglement in frogs"))   # off-corpus → [] (honest!)

## Section 8 — Grounded answering with sources

Two rules in the system prompt do most of the anti-hallucination work:
**answer only from context** and **cite sources like [source]**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

docs_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the context below. "
               "Cite the source of each fact in brackets, like [policy.pdf]. "
               "If the context does not contain the answer, say you don't have that information."),
    ("human", "Context:\n{context}\n\nQuestion: {user_input}"),
])

def format_hits(hits: list[dict]) -> str:
    return "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)

def answer_from_docs(question: str) -> str:
    hits = retrieve(question)
    if not hits:
        return "I couldn't find anything about that in the documents."
    result = (docs_prompt | llm).invoke({
        "user_input": question,
        "context": format_hits(hits),
    })
    return result.content

print(answer_from_docs("how much does car repair cost?"))
print()
print(answer_from_docs("what is the meaning of life?"))   # honest miss

## Section 9 — Rebuild the agent, upgrade the `docs` node

This is your Week 2 Class 3 agent, condensed — **identical** except for one thing: `docs_node`
now calls `retrieve()` instead of reading a hardcoded `context_chunks` string.

> Have your Week 2 notebook open with the agent already built? Then just replace its
> `docs_node` with the version below and re-compile the graph — that's the whole upgrade.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    intent: str
    context_chunks: str   # kept for Week 2 compatibility — docs no longer needs it!

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the user message as exactly one word: chat, calculate, research, or docs.\n"
               "- chat: general conversation, opinions\n"
               "- calculate: any arithmetic the user wants computed\n"
               "- research: needs a current fact to look up online\n"
               "- docs: asks about \"the documents\", \"the PDF\", our data, policies, or stored knowledge"),
    ("human", "{user_input}"),
])

def classify_node(state: AgentState) -> dict:
    user_text = state["messages"][-1].content
    result = (classify_prompt | llm).invoke({"user_input": user_text})
    intent = result.content.strip().lower()
    for known in ("calculate", "research", "docs"):
        if known in intent:
            return {"intent": known}
    return {"intent": "chat"}

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly, helpful assistant. Answer clearly in under 5 sentences."),
    ("human", "{user_input}"),
])

def chat_node(state: AgentState) -> dict:
    result = (chat_prompt | llm).invoke({"user_input": state["messages"][-1].content})
    return {"messages": [AIMessage(content=result.content)]}

calc_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are given a math question and its computed answer. State the answer clearly in one sentence."),
    ("human", "Question: {user_input}\nComputed result: {result}"),
])

def calculate_node(state: AgentState) -> dict:
    user_text = state["messages"][-1].content
    expr = "".join(ch for ch in user_text if ch in "0123456789+-*/(). ")
    try:
        computed = str(eval(expr.strip())) if expr.strip() else "unable to parse"
    except Exception as e:
        computed = f"Error: {e}"
    result = (calc_prompt | llm).invoke({"user_input": user_text, "result": computed})
    return {"messages": [AIMessage(content=result.content)]}

research_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using the search result below. Mention it comes from a web search. "
               "If the result doesn't answer the question, say so honestly."),
    ("human", "Question: {user_input}\n\nSearch result:\n{search_result}"),
])

def research_node(state: AgentState) -> dict:
    user_text = state["messages"][-1].content
    try:
        from langchain_community.tools import DuckDuckGoSearchRun
        search_result = DuckDuckGoSearchRun().run(user_text)[:800]
    except Exception as e:
        search_result = f"(web search unavailable: {e})"
    result = (research_prompt | llm).invoke({"user_input": user_text, "search_result": search_result})
    return {"messages": [AIMessage(content=result.content)]}

# ── THE WEEK 3 UPGRADE ──────────────────────────────────────────────
def docs_node(state: AgentState) -> dict:
    """Week 2: read a hardcoded string. Week 3: retrieve from Qdrant."""
    question = state["messages"][-1].content
    hits = retrieve(question, k=3)
    if not hits:
        return {"messages": [AIMessage(content="I couldn't find anything about that in the documents.")]}
    result = (docs_prompt | llm).invoke({
        "user_input": question,
        "context": format_hits(hits),
    })
    sources = ", ".join(sorted({h["source"] for h in hits}))
    return {"messages": [AIMessage(content=f"{result.content}\n\n(retrieved from: {sources})")]}
# ────────────────────────────────────────────────────────────────────

def route(state: AgentState) -> str:
    return state["intent"]

graph = StateGraph(AgentState)
for name, node in [("classify", classify_node), ("chat", chat_node),
                   ("calculate", calculate_node), ("research", research_node),
                   ("docs", docs_node)]:
    graph.add_node(name, node)
graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route,
    {"chat": "chat", "calculate": "calculate", "research": "research", "docs": "docs"})
for name in ("chat", "calculate", "research", "docs"):
    graph.add_edge(name, END)

agent = graph.compile()
print("Agent compiled: classify → chat / calculate / research / docs(Qdrant)")

## Section 10 — Test all four intents

In [ ]:
def ask(prompt: str):
    result = agent.invoke({
        "messages": [HumanMessage(content=prompt)],
        "intent": "", "context_chunks": "",
    })
    print(f"\n══ '{prompt}'  →  intent: {result['intent']}")
    print(result["messages"][-1].content[:400])

ask("hey, how's it going?")                          # chat
ask("what is 23 * 19 - 7?")                          # calculate
ask("what do the documents say about refunds?")      # docs → Qdrant retrieval
ask("what's in our data about the world cup groups?")# docs → Qdrant retrieval

In [ ]:
# research reaches the live web — may be slow or rate-limited in Colab
ask("latest news about the 2026 World Cup")

**What just happened:** four intents, four engines behind one chat box — and the `docs`
answers now change when you change the *collection*, not the *code*. Knowledge became data.

### 🔹 Mini-exercise (5 min)

Ask a docs question that is **off-corpus** ("what do the documents say about dinosaurs?").
Confirm the threshold produces the honest "couldn't find" answer instead of a hallucination.
Then lower `SCORE_THRESHOLD` to 0.05 and watch what goes wrong.

## Section 11 — Debugging checklist

| Symptom | Fix |
|---------|-----|
| `docs` answers feel generic | Print `retrieve(question)` — are the right chunks coming back? Scores? |
| Right doc, diluted answer | Chunks too big — re-chunk smaller (Week 2 Section 13) and re-upsert |
| Answer split across chunks | Raise `k`, or re-chunk with more overlap |
| Off-corpus questions hallucinate | Threshold too low — raise `SCORE_THRESHOLD` |
| Real questions get "couldn't find" | Threshold too high, or your data genuinely lacks it |
| docs questions route to chat | Improve the classify prompt's docs description |

## Section 12 — Lab deliverable & homework

**Week 3 deliverable — Retrieval-Enabled LangGraph Agent:**

- [ ] Your **own** Week 2 chunks (not the fallback) embedded and upserted with payloads
      (`text`, `source`, plus `page` / `price_gbp` where you have them)
- [ ] `docs` node retrieves top-3 live from Qdrant — no hardcoded context anywhere
- [ ] Score threshold working: off-corpus questions get an honest miss
- [ ] At least one **filtered** query working (by source or price)
- [ ] Demo (2–3 min): one prompt per intent · a grounded answer **with sources shown** ·
      an honest miss · a filtered search

**Homework:**

1. **Merge everything** — load your blog posts, World Cup standings, book dataset, AND PDF
   chunks into one collection with distinct `source` payloads. How many points total?
2. **Threshold tuning** — collect 5 answerable + 5 unanswerable questions; find the threshold
   that separates them best on your data. Report false accepts/rejects.
3. **Stretch: persistence** — swap `":memory:"` for `QdrantClient(path="./qdrant_data")` and
   confirm your collection survives a runtime restart.
4. **Stretch: a 5th intent** — if you built one in Week 2 homework, port it into this agent.

**Next week:** RAG evaluation, hybrid (BM25 + vectors) retrieval, re-ranking, and the final
Gradio showcase. Your Qdrant collection is the knowledge base for the final bot — keep it.